# U3.L1 — CFM from Scratch: The Payoff

**Session U3.L1** (lab, joint). Companion to the U3.L1 lecture notes (<https://adricortes.github.io/flow-matching-lectures/U3/U3-L1-notes.html>), which carry the
reasoning; this notebook is the part you execute.

You write the conditional flow matching (CFM) training loop, train it on the data and under the budget of your
U2.L2 FFJORD run, sample from it with the course's own ODE solver, and put the two receipts side by side.

### How to read this notebook

Clone the public repository (<https://github.com/adricortes/flow-matching-labs>) and open this notebook from
`labs/`. Your U2.L2 evidence (`pain_evidence/` with `pair.json`) must sit where U2.L2 wrote it, at the root of
the clone. If you have no U2.L2 evidence, the notebook uses the shipped reference pair and says so.

This lab needs two packages more than U2.L2, TorchCFM and POT (the only dependency its import needs), installed
without TorchCFM's declared dependencies: `pip install --no-deps torchcfm==1.0.7 POT==0.9.7.post1`. Without it, the first code cell stops
and prints that command.

Cells marked **TODO** are yours. Everything else is given code: run it, read it, go on. Prompt cells ask for two
to four sentences in your own words. Checkpoints pass or say what is wrong.

## Part 0 — Prelude and the promissory note

| File | What it is |
|---|---|
| `common/toy2d.py` | the datasets of U1.L1, unchanged |
| `common/odesolve.py` | the course's ODE solvers (`odeint`, `SolverOutput` with `.nfe`) |
| `common/painplots.py` | the measurement helper of U2.L2 (v1.1); its contract is frozen |
| `common/payoff.py` | today's comparison helper, **given complete** |
| `configs/u2l2-budget.yaml` | the frozen budget of U2.L2, inherited field by field |
| `configs/u2l2-rescue.yaml` | the rescue budget, used for a dataset whose U2.L2 run used it |
| `labs/u2l2/reference/`, `labs/u3l1/reference/` | the reference pair: CNF evidence and CFM evidence of one computer |

The budget's solver settings (dopri5, rtol 1e-5, atol 1e-7) are a training setting of the CNF. CFM training uses
no solver; CFM uses solvers only after training, for sampling and for the evaluation sweep.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import copy
import inspect
import json
import math
import random
import shutil
import statistics
import time
import warnings

import matplotlib.pyplot as plt
from IPython.display import display
import mlflow               # Part 6 needs it: a missing install stops here, not after the training
import numpy as np
import torch
import torch.nn as nn
import yaml
from torchdiffeq import odeint, odeint_adjoint

from common import odesolve, painplots, payoff
from common.toy2d import EXTENT, checkerboard, density_heatmap, in_checkerboard_support, two_moons, warped_grid

torch.set_num_threads(1)    # one thread, as in U2.L2: wall-clock values stay comparable
warnings.filterwarnings("ignore", message="Converting a tensor with requires_grad")

GENERATORS = {"two-moons": two_moons, "checkerboard": checkerboard}
LOG2PI = math.log(2 * math.pi)
EVIDENCE = ROOT / "pain_evidence"                # your U2.L2 evidence
CFM_EVIDENCE = ROOT / "cfm_evidence"             # what this notebook writes
REF_CNF = ROOT / "labs" / "u2l2" / "reference" / "pain_evidence"
REF_CFM = ROOT / "labs" / "u3l1" / "reference" / "cfm_evidence"
HEAD_SCALE = 0.1
RESULTS = globals().get("RESULTS", {})           # one entry per dataset; running this cell again keeps them

def set_all_seeds(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

try:                        # Part 5 needs it: a missing install stops here, not after the training
    from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
except ImportError as err:
    raise ImportError("TorchCFM is missing. Install it with:  "
                      "pip install --no-deps torchcfm==1.0.7 POT==0.9.7.post1") from err

# The budget follows YOUR U2.L2 run, dataset by dataset: U2.L2 let each dataset use its own tier.
def own_budget(ds: str) -> str:
    meta = EVIDENCE / ds / "meta.json"
    return json.loads(meta.read_text())["budget_version"] if meta.exists() else painplots.BUDGET_VERSION

def config_for(budget: str) -> dict:
    name = "u2l2-rescue.yaml" if budget == painplots.BUDGET_VERSION_RESCUE else "u2l2-budget.yaml"
    return yaml.safe_load((ROOT / "configs" / name).read_text())

CFG = {ds: config_for(own_budget(ds)) for ds in ("two-moons", "checkerboard")}
cfg = CFG["two-moons"]      # Parts 1 and 5 work on two-moons

# The pair id of THIS computer, read from your U2.L2 pair.json and never overwritten. If there is none (or it
# belongs to another computer), a new id is minted beside today's evidence; it pairs with nothing from U2.L2.
pair_file = EVIDENCE / "pair.json"
pair_data = json.loads(pair_file.read_text()) if pair_file.exists() else {}
PAIR_ID = (pair_data["pair_id"] if pair_data.get("machine_id") == painplots.machine_id()
           else painplots.mint_pair_id(CFM_EVIDENCE))
print("torch", torch.__version__, "| machine_id", painplots.machine_id(), "| pair_id", PAIR_ID)
for ds, c in CFG.items():
    print(f"[{ds}] budget {c['budget_version']}")

# Defined here, not inside a checkpoint, so that no later cell depends on Checkpoints 1.1 and 1.2 having run.
# After a restart, Parts 3 to 6 still need the Part 2 training cells: the fields live in RESULTS, in memory.
train_x = GENERATORS["two-moons"](cfg["data"]["n_train"], cfg["data"]["seed_train"])

def reference_field() -> nn.Module:
    """The reference CFM field of two-moons (width 64): the fixed input of Checkpoints 1.2, 3.1 and 3.2."""
    f = Field(64, 3)       # the class is in the next code cell; it is looked up when this function runs
    f.load_state_dict(torch.load(ROOT / "common" / "weights" / "u3l1-two-moons.pt", map_location="cpu")["state_dict"])
    return f

### The given U2.L2 code (`ffjord-scratch`)

The cell below is the U2.L2 solution, given: the field class and the augmented ODE that computes
$\log p_1(x_1)$. Today it is used **at evaluation only**, to measure a test NLL. Read the docstrings; the U2.T2
notes carry the derivation.

In [ ]:
class Field(nn.Module):
    """The learned field u_t^theta(x): R^2 -> R^2, an MLP with time concatenated.

    The solver calls it as f(t, x), time first (U0.T5), with t a scalar tensor. The
    network sees t as a (B,) column beside x: the course convention for time in a
    network signature. `depth` hidden layers of width `hidden`, tanh between them.
    """

    def __init__(self, hidden: int = 64, depth: int = 3, zero_head: bool = True) -> None:
        super().__init__()
        layers, width_in = [], 3
        for _ in range(depth):
            layers += [nn.Linear(width_in, hidden), nn.Tanh()]
            width_in = hidden
        layers.append(nn.Linear(hidden, 2))
        self.net = nn.Sequential(*layers)
        if zero_head:
            # The U0.T2 defaults, item 3: zero-init the last layer, so u = 0 at the start.
            nn.init.zeros_(self.net[-1].weight)
            nn.init.zeros_(self.net[-1].bias)

    def forward(self, t: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        tb = t.to(x.dtype).expand(x.shape[0])                  # (B,)
        return self.net(torch.cat([x, tb[:, None]], dim=1))


def exact_divergence(u: torch.Tensor, x: torch.Tensor, create_graph: bool) -> torch.Tensor:
    """div u = sum_i d u_i / d x_i, one autograd call per coordinate.

    In d = 2 this costs two backward passes and is the deterministic ground truth of
    every numerical check and every evaluation sweep in this lab.
    """
    return sum(
        torch.autograd.grad(u[:, i].sum(), x, create_graph=create_graph, retain_graph=True)[0][:, i]
        for i in range(x.shape[1]))


class AugmentedDynamics(nn.Module):
    """d/dt [x; Delta] = [u_t^theta(x); -div u_t^theta(x)].

    The state is `[x (2), delta_logp (1)]`, plus two integral-loss slots when
    `regularize` is on (the RNODE penalties of U2.L2). The solve runs from t = 1
    (data) to t = 0 (base) with delta_logp(1) = 0. Then, with Delta(t) = log p_t(x_t) - log p_1(x_1),
    delta_logp(0) = Delta(0) and log p_1(x_1) = log p_0(x_0) - delta_logp(0).

    HUTCHINSON, FIXED PROBE. `before_solve(x)` draws one Rademacher probe per example,
    ONCE per minibatch likelihood solve. The probe is stored and reused by every call of the
    solver and by every recomputation the adjoint performs. It is never drawn inside
    `forward`. `self.nfe` counts calls of the field (a vector-Jacobian product made inside a
    call is not another call).
    """

    def __init__(self, field: nn.Module, divergence: str = "hutchinson", regularize: bool = False,
                 weights: tuple[float, float] = (0.0, 0.0), probe_seed: int = 0) -> None:
        super().__init__()
        assert divergence in ("hutchinson", "exact")
        self.field, self.divergence, self.regularize, self.weights = field, divergence, regularize, weights
        self.nfe = 0
        self.probe = None
        self.audit = None            # a list of probe checksums, when the audit is on
        self.probe_gen = torch.Generator().manual_seed(probe_seed)

    def before_solve(self, x: torch.Tensor) -> None:
        """Sample the probe once for the coming likelihood solve."""
        self.nfe = 0                                          # the count of calls restarts with every solve
        bits = torch.randint(0, 2, tuple(x.shape), generator=self.probe_gen)
        self.probe = (bits * 2 - 1).to(device=x.device, dtype=x.dtype)    # Rademacher: +1 or -1

    def forward(self, t: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
        self.nfe += 1
        if self.audit is not None:
            self.audit.append(float((self.probe.double() * torch.arange(1, self.probe.numel() + 1,
                                     dtype=torch.float64).reshape(self.probe.shape)).sum()))
        create_graph = torch.is_grad_enabled()      # True in the adjoint pass and in direct backprop
        x = state[:, :2]
        with torch.enable_grad():
            if not x.requires_grad:
                x = x.detach().requires_grad_(True)     # forward solve: a fresh leaf
            # In the adjoint pass `state` already carries a graph: x must stay connected to it.
            u = self.field(t, x)
            if self.divergence == "exact":
                div = exact_divergence(u, x, create_graph)
                vjp = None
            else:
                vjp = torch.autograd.grad(u, x, self.probe, create_graph=create_graph)[0]   # v^T J
                div = (vjp * self.probe).sum(1)                                              # v^T J v
        out = [u, -div[:, None]]
        if self.regularize:
            if vjp is None:
                raise ValueError("the regularizer reuses the probe's product: use the Hutchinson mode")
            out += [-(u ** 2).sum(1, keepdim=True), -(vjp ** 2).sum(1, keepdim=True)]
        return torch.cat(out, dim=1)


def solve_augmented(dyn: AugmentedDynamics, x1: torch.Tensor, *, rtol: float, atol: float,
                    method: str = "dopri5", adjoint: bool = True, **options) -> torch.Tensor:
    """Integrate the augmented state from t = 1 to t = 0; returns the state at t = 0 (given).

    The probe must already be set (`before_solve`). `adjoint=True` is the course default
    for training; `adjoint=False` differentiates through the solver steps instead.
    """
    extra = 2 if dyn.regularize else 0
    state1 = torch.cat([x1, x1.new_zeros(x1.shape[0], 1 + extra)], dim=1)
    t_span = x1.new_tensor([1.0, 0.0])
    solve = odeint_adjoint if adjoint else odeint
    return solve(dyn, state1, t_span, rtol=rtol, atol=atol, method=method, **options)[-1]


def neg_log_likelihood(dyn: AugmentedDynamics, x1: torch.Tensor, *, rtol: float, atol: float,
                       adjoint: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
    """-log p_1(x_1) per example, and the final state. One fresh probe per call."""
    dyn.before_solve(x1)
    state0 = solve_augmented(dyn, x1, rtol=rtol, atol=atol, adjoint=adjoint)
    x0, delta_logp = state0[:, :2], state0[:, 2]
    logp0 = -0.5 * (x0 ** 2).sum(1) - LOG2PI              # N(0, I_2) at x_0
    log_p1 = logp0 - delta_logp                           # the sign contract: minus Delta(0)
    return -log_p1, state0


@torch.no_grad()
def evaluate_nll(field: nn.Module, x: torch.Tensor, *, rtol: float, atol: float) -> dict:
    """Test NLL with the EXACT divergence on one batch: isolates solver error from probe noise.

    It goes through `neg_log_likelihood`, so it uses your sign contract. The number of calls of the field
    is read from the dynamics after the solve."""
    dyn = AugmentedDynamics(field, divergence="exact")
    t0 = time.perf_counter()
    nll, _ = neg_log_likelihood(dyn, x, rtol=rtol, atol=atol, adjoint=False)
    wall = time.perf_counter() - t0
    return {"nll": float(nll.mean()), "nfe": dyn.nfe, "wall_s": wall}


def warmup_cosine(step: int, total: int, warmup: int) -> float:
    """Multiplier on the base learning rate: the U0.T2 schedule, stepped per iteration."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))


@torch.no_grad()
def make_log_prob(field: nn.Module, rtol: float = 1e-5):
    def log_prob(pts: torch.Tensor) -> torch.Tensor:
        dyn = AugmentedDynamics(field, "exact")
        dyn.before_solve(pts)
        s0 = solve_augmented(dyn, pts, rtol=rtol, atol=rtol * 1e-2, adjoint=False)
        return -0.5 * (s0[:, :2] ** 2).sum(1) - LOG2PI - s0[:, 2]
    return log_prob


@torch.no_grad()
def make_generator(field: nn.Module):
    def gen(x0: torch.Tensor) -> torch.Tensor:
        f = lambda t, x: field(t, x)    # noqa: E731
        return odeint(f, x0, x0.new_tensor([0.0, 1.0]), rtol=1e-5, atol=1e-7, method="dopri5")[-1]
    return gen

### Your evidence from U2.L2

The cell validates each evidence directory and prints which comparison applies today: **own** (your U2.L2 run
and your CFM run of today pair on this computer) or **reference** (the shipped pair). The pairing key is
`machine_id + pair_id + budget_version`; inside the evidence contract, wall-clock time is compared only inside one pair.

In [ ]:
for ds in ("two-moons", "checkerboard"):
    d = EVIDENCE / ds
    problems = painplots.validate_evidence_dir(d, expected_session="U2.L2") if d.exists() else ["no directory"]
    assert not painplots.validate_evidence_dir(REF_CNF / ds, expected_session="U2.L2"), "the shipped reference is damaged"
    if problems:
        print(f"[{ds}] your U2.L2 evidence is not usable ({problems[0].replace(str(ROOT) + '/', '')})"
              " -> expected mode: reference")
        continue
    meta = json.loads((d / "meta.json").read_text())
    differ = [k for k, mine in (("machine_id", painplots.machine_id()), ("pair_id", PAIR_ID),
                                ("budget_version", CFG[ds]["budget_version"])) if meta[k] != mine]
    print(f"[{ds}] your U2.L2 evidence is valid -> expected mode: "
          + ("own" if not differ else f"reference ({', '.join(differ)} differs from this computer's run)"))
print("The rule (evidence contract): wall-clock is compared only between two runs that share machine_id, pair_id and budget_version.")

### The promissory note

Today's loss regresses $u_t^\theta$ onto the *conditional* velocity, a target you can compute, instead of the
marginal velocity $u_t$ defined in U3.T1. U3.T2 proves exactly how the two squared objectives are related.
Today we collect the evidence that makes that theorem worth proving.

## Part 1 — The loop

The CondOT path of U3.T1: $x_t = t\,x_1 + (1 - t)\,x_0$ with $x_0 \sim p_0 = \mathcal{N}(0, I_2)$ and
$x_1 \sim q$, and its conditional velocity $u_t(x_t \mid x_1) = x_1 - x_0$. The loss of today, with every
sampling law written on the expectation:

$$\mathcal{L}_{\mathrm{CFM}}(\theta) = \mathbb{E}_{t \sim \mathrm{Unif}[0,1],\; x_1 \sim q,\; x_0 \sim p_0}
\big\| u_t^\theta(x_t) - (x_1 - x_0) \big\|^2 .$$

This loss is exact: every term in it can be computed. What is deferred to U3.T2 is why its gradients are the
right ones.

In [ ]:
def sample_batch(train_x: torch.Tensor, batch: int, gen: torch.Generator):
    """One CFM minibatch on the CondOT path (TODO 1.1). Returns (t, x0, x1, xt, target).

    t ~ Unif[0, 1] has shape (B,): the code-level FM arrow, t = 0 noise and t = 1 data.
    x1 ~ q is a row of the training set, x0 ~ N(0, I_2), both (B, 2). Every draw comes from `gen`.
    """
    idx = torch.randint(0, train_x.shape[0], (batch,), generator=gen)
    x1 = train_x[idx]                                                # (B, 2), data
    # TODO 1.1: in this order, every draw from `gen`, in the dtype of x1:
    #   x0     ~ N(0, I_2), shape (B, 2)                    (torch.randn)
    #   t      ~ Unif[0, 1], shape (B,)                     (torch.rand)
    #   xt     = the CondOT point of the U3.T1 notes at time t (t = 0 noise, t = 1 data);
    #            t is (B,) and x0, x1 are (B, 2): mind the broadcast
    #   target = the conditional velocity of the CondOT path
    raise NotImplementedError
    return t, x0, x1, xt, target

In [ ]:
# Checkpoint 1.1: the minibatch
t, x0, x1, xt, target = sample_batch(train_x, 4, torch.Generator().manual_seed(0))
assert t.shape == (4,), f"t must have shape (B,) = (4,), got {tuple(t.shape)}"
assert x0.shape == x1.shape == xt.shape == target.shape == (4, 2), "x0, x1, xt and target must be (B, 2)"
assert torch.allclose(x1, torch.tensor([[-0.4694811999797821, 1.447319746017456], [-1.5642706155776978, 1.686523199081421], [-0.7988876104354858, 1.6752772331237793], [-2.9621710777282715, 0.981397271156311]])), "x1 differs: keep the given index draw first"
assert torch.allclose(x0, torch.tensor([[-2.1787893772125244, 0.5684312582015991], [-1.0845223665237427, -1.3985954523086548], [0.40334683656692505, 0.8380263447761536], [-0.7192575931549072, -0.40334352850914]])), "x0 differs: draw x0 right after the index, with torch.randn and gen"
assert torch.allclose(t, torch.tensor([0.6816085577011108, 0.9151939749717712, 0.39709991216659546, 0.8741558790206909])), "t differs: draw t after x0, with torch.rand(B) and gen"
assert torch.allclose(xt, torch.tensor([[-1.0137102603912354, 1.1674891710281372], [-1.5235850811004639, 1.4248865842819214], [-0.0740603655576706, 1.1704986095428467], [-2.6799137592315674, 0.8071357607841492]]), atol=1e-6), \
    "xt is not the CondOT point: at t = 0 it must equal x0 (noise), at t = 1 it must equal x1 (data)"
assert torch.allclose(target, torch.tensor([[1.70930814743042, 0.8788884878158569], [-0.4797482490539551, 3.0851187705993652], [-1.2022345066070557, 0.8372508883476257], [-2.2429134845733643, 1.3847408294677734]]), atol=1e-6), "the target is the CondOT conditional velocity, x1 - x0"
tb = sample_batch(train_x, 100_000, torch.Generator().manual_seed(1))
assert abs(float(tb[0].mean()) - 0.5) < 0.01 and float(tb[0].min()) >= 0 and float(tb[0].max()) <= 1, "t must be Unif[0, 1]"
assert abs(float(tb[1].std()) - 1) < 0.01, "x0 must be standard normal"
print("Checkpoint 1.1 passed.")

In [ ]:
def cfm_loss(field: nn.Module, t: torch.Tensor, xt: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """The minibatch estimate of L_CFM (TODO 1.2): squared norm over coordinates, mean over the batch."""
    # TODO 1.2: call the field as field(t, xt), time first, with t of shape (B,).
    # Squared Euclidean norm of the residual over the coordinates, then the mean over the batch.
    raise NotImplementedError

In [ ]:
# Checkpoint 1.2: the loss, on your sample_batch and your cfm_loss
# (a) At initialization the head is zero, so u = 0 and the loss is E||x1 - x0||^2 = E||x1||^2 + 2.
pop = float((train_x.double() ** 2).sum(1).mean()) + 2.0
f0 = Field(cfg["model"]["hidden"], cfg["model"]["depth"])
gen = torch.Generator().manual_seed(cfg["seed"])
vals = []
for _ in range(200):
    tb, _, _, xb, yb = sample_batch(train_x, cfg["train"]["batch"], gen)
    vals.append(float(cfm_loss(f0, tb, xb, yb)))
mean, se = statistics.fmean(vals), statistics.stdev(vals) / math.sqrt(len(vals))
print(f"loss at init: {mean:.3f} over 200 minibatches; population value {pop:.3f}")
assert abs(mean - pop) < 4 * se, \
    f"the mean loss at init is {mean:.3f}, expected {pop:.3f}: sum the squares over the coordinates, then average over the batch"
# (b) The reference CFM weights, your loss, a fixed minibatch: the stored value.
fr = reference_field()
tb, _, _, xb, yb = sample_batch(train_x, 256, torch.Generator().manual_seed(5))
val = float(cfm_loss(fr, tb, xb, yb))
assert abs(val - 5.282547473907471) < 1e-4, (
    f"loss {val:.4f}, expected {5.2825}"
    + (": the target has the wrong sign" if abs(val - 14.251721382141113) < 1e-3 else ""))
print("Checkpoint 1.2 passed.")

In [ ]:
def make_field(cfg: dict) -> nn.Module:
    """The field of the inherited budget (TODO 1.3): the U2.L2 `Field` class with the budget's width and depth, tanh, zero head."""
    m = cfg["model"]
    assert m["activation"] == "tanh", "the budget fixes tanh, and Field implements tanh"
    # TODO 1.3: build the given U2.L2 `Field` with the width, the depth and the zero head of `cfg`.
    raise NotImplementedError

In [ ]:
# Checkpoint 1.3: the field of the budget, beside the FFJORD field of U2.L2
f_cfm = make_field(cfg)
f_cnf = Field(cfg["model"]["hidden"], cfg["model"]["depth"])
n_cfm, n_cnf = (sum(p.numel() for p in f.parameters()) for f in (f_cfm, f_cnf))
print(f"parameters: CFM field {n_cfm}, FFJORD field {n_cnf}")
assert n_cfm == n_cnf, "the CFM field must be the budget's field: same width, same depth"
assert torch.count_nonzero(f_cfm(torch.rand(5), torch.randn(5, 2))) == 0, "the budget zero-initializes the head"
print("Checkpoint 1.3 passed.")

### The gradient-check ritual

The standing ritual of the course, CFM instance: autograd against a central finite difference, in float64,
along one fixed direction of the **first** layer's weight, on ONE frozen minibatch (nothing is drawn between the
$+h$ and the $-h$ evaluations).

The budget's head is zero, so every parameter upstream of it has gradient exactly 0, and a check there compares 0
with 0. The cell shows that, then moves the head by a fixed, stated amount (entries
$0.1 \cdot \mathcal{N}(0, 1)$, seed 0) and checks again.

In [ ]:
@torch.no_grad()
def move_head(field: nn.Module, scale: float = HEAD_SCALE, seed: int = 0) -> nn.Module:
    """A copy of `field` whose zero head is replaced by scale * N(0, 1) entries (a fixed, stated move).

    With the head at zero every parameter upstream of it has gradient exactly 0, and a gradient
    check there compares 0 with 0 (a course lesson: a zero head makes every upstream check vacuous). Moving the
    head first makes it a test.
    """
    g = copy.deepcopy(field)
    gen = torch.Generator().manual_seed(seed)
    head = g.net[-1]
    head.weight.copy_(scale * torch.randn(head.weight.shape, generator=gen))
    head.bias.copy_(scale * torch.randn(head.bias.shape, generator=gen))
    return g


def directional_check(field: nn.Module, batch: tuple, *, h: float = 1e-6, seed: int = 0) -> dict:
    """Autograd against a central difference, in float64, along one fixed direction of net[0].weight.

    `batch` is ONE frozen minibatch (t, x0, x1, xt, target): nothing is drawn between the +h and
    the -h evaluations. The direction is a unit vector of fixed seed in the FIRST layer, the
    tensor farthest from the head.
    """
    f = copy.deepcopy(field).double()
    t, _, _, xt, target = (b.double() for b in batch)
    p = f.net[0].weight
    v = torch.randn(p.shape, generator=torch.Generator().manual_seed(seed), dtype=torch.float64)
    v = v / v.norm()
    grad = torch.autograd.grad(cfm_loss(f, t, xt, target), p)[0]
    autodiff = float((grad * v).sum())
    with torch.no_grad():
        p.add_(h * v)
        plus = float(cfm_loss(f, t, xt, target))
        p.add_(-2 * h * v)
        minus = float(cfm_loss(f, t, xt, target))
        p.add_(h * v)
    central = (plus - minus) / (2 * h)
    rel = abs(autodiff - central) / max(abs(autodiff), abs(central), 1e-300)
    return {"autodiff": autodiff, "central": central, "rel_err": rel}


batch = sample_batch(train_x, 64, torch.Generator().manual_seed(7))
torch.manual_seed(cfg["seed"])               # the hidden layers' default init: seeded, so the reading repeats
f = make_field(cfg)
zero = directional_check(f, batch)
moved = directional_check(move_head(f), batch)
print(f"zero head : autodiff {zero['autodiff']:.3e}  central {zero['central']:.3e}   (a comparison of 0 with 0)")
print(f"moved head: autodiff {moved['autodiff']:.10f}  central {moved['central']:.10f}  rel. error {moved['rel_err']:.1e}")
print("reference computer, full budget: autodiff 0.0398358960")
assert zero["autodiff"] == 0.0, "with a zero head the first layer's gradient is exactly 0"
assert abs(moved["autodiff"]) > 1e-4, "the check measured nothing: the directional derivative is 0"
assert moved["rel_err"] < 1e-6, "autograd and the central difference disagree: is fresh randomness drawn inside cfm_loss?"
print("Checkpoint 1.4 (ritual) passed.")

In [ ]:
# Checkpoint 1.5: one training step on the CPU, no solver in it
fs = make_field(cfg)
opt = torch.optim.AdamW(fs.parameters(), lr=cfg["opt"]["lr"])
gen = torch.Generator().manual_seed(0)
times = []
for _ in range(60):
    s0 = time.perf_counter()
    tb, _, _, xb, yb = sample_batch(train_x, cfg["train"]["batch"], gen)
    loss = cfm_loss(fs, tb, xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
    times.append(time.perf_counter() - s0)
step_ms = 1e3 * statistics.median(times[10:])
print(f"one step: {step_ms:.2f} ms (the reference computer: {1.02} ms)")
assert step_ms < 50, f"one step takes {step_ms:.1f} ms: more than 50 ms on a CPU means something else runs inside the step"
print("Checkpoint 1.5 passed.")

## Part 2 — Train under the inherited budget

`train_cfm` is given: the seed, the minibatch generator, the optimizer and the schedule of the U2.L2 `train_run`,
with your `sample_batch` and `cfm_loss` inside. It logs every iteration from iteration 0. The two NFE columns
record 0 solver calls, because no ODE solver runs inside this loss. The one direct call of the field per step
is `field_evals_train = 1`, logged in Part 6 under its own name, never as NFE.

In [ ]:
def train_cfm(cfg: dict, dataset: str, *, seed: int | None = None, iters: int | None = None,
              batch_fn=None, verbose: bool = True):
    """Train under the inherited budget, instrumented from iteration 0. Returns (field, log).

    The same seed, minibatch generator, optimizer and schedule as the U2.L2 `train_run`. The step
    time covers the minibatch draw, the loss, the backward pass and the update. The two NFE columns
    record 0: no ODE solver runs inside this loss. The one direct call of the field per step is
    `field_evals_train`, logged elsewhere under its own name.
    """
    seed = cfg["seed"] if seed is None else seed
    iters = cfg["train"]["iters"] if iters is None else iters
    batch_fn = sample_batch if batch_fn is None else batch_fn
    set_all_seeds(seed)
    train_x = GENERATORS[dataset](cfg["data"]["n_train"], cfg["data"]["seed_train"])
    field = make_field(cfg)
    opt = torch.optim.AdamW(field.parameters(), lr=cfg["opt"]["lr"], betas=tuple(cfg["opt"]["betas"]),
                            weight_decay=cfg["opt"]["weight_decay"])
    warm = max(1, round(cfg["sched"]["warmup_frac"] * iters))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: warmup_cosine(i, iters, warm))
    batch_gen = torch.Generator().manual_seed(seed)
    log = painplots.TrainLog()
    for it in range(1, iters + 1):
        t0 = time.perf_counter()
        t, x0, x1, xt, target = batch_fn(train_x, cfg["train"]["batch"], batch_gen)
        loss = cfm_loss(field, t, xt, target)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        step_s = time.perf_counter() - t0
        if not math.isfinite(loss.item()):
            raise FloatingPointError(f"loss is {loss.item()} at iteration {it}")
        log.add(step_s, 0, 0, loss.item())               # solver NFE: 0 and 0, there is no solve
        if verbose and it % 500 == 0:
            print(f"  [{dataset}] it {it:5d}  loss {statistics.fmean(log.loss[-100:]):.4f}  "
                  f"s/it {statistics.fmean(log.step_s[-100:]):.5f}", flush=True)
    return field, log

While it runs: open your U2.L2 `pain_wallclock.csv`. Note the units.

In [ ]:
DATASET = "two-moons"
field, log = train_cfm(CFG[DATASET], DATASET)
RESULTS[DATASET] = {"field": field, "log": log}
last = statistics.fmean(log.loss[-300:])
print(f"{DATASET}: {len(log)} iterations in {sum(log.step_s):.1f} s; mean loss of the last 300: {last:.3f}")

In [ ]:
DATASET = "checkerboard"
field, log = train_cfm(CFG[DATASET], DATASET)
RESULTS[DATASET] = {"field": field, "log": log}
last = statistics.fmean(log.loss[-300:])
print(f"{DATASET}: {len(log)} iterations in {sum(log.step_s):.1f} s; mean loss of the last 300: {last:.3f}")

In [ ]:
# Checkpoint 2: the last-300 loss of each run, against five reference seeds (budget U2L2-B1)
windows = {"two-moons": (4.96340058962504, 5.059247481028239),
           "checkerboard": (7.135394296646118, 7.201600672403972)}
for ds, (lo, hi) in windows.items():
    last = statistics.fmean(RESULTS[ds]["log"].loss[-300:])
    if CFG[ds]["budget_version"] != painplots.BUDGET_VERSION:
        print(f"{ds}: {last:.3f} (rescue budget: no reference window)")
        continue
    print(f"{ds}: {last:.3f}, reference seeds {lo:.3f} to {hi:.3f}")
    assert lo - 0.05 <= last <= hi + 0.05, f"{ds}: the final loss {last:.3f} is outside [{lo - 0.05:.3f}, {hi + 0.05:.3f}]"
fig, ax = plt.subplots(figsize=(8, 4))
for ds, color in (("two-moons", "#003366"), ("checkerboard", "#d97a1f")):
    loss = RESULTS[ds]["log"].loss
    ax.plot(range(10, len(loss) + 1, 10), [statistics.fmean(loss[i:i + 10]) for i in range(0, len(loss), 10)],
            color=color, label=ds)
ax.set_xlabel("training iteration"); ax.set_ylabel("minibatch loss (mean of 10)"); ax.legend(frameon=False)
plt.show()
print("Checkpoint 2 passed." if all(c["budget_version"] == painplots.BUDGET_VERSION for c in CFG.values())
      else "Checkpoint 2: the rescue-budget dataset was skipped (no reference window); the others passed.")

The learned two-moons model, in the visual language of U1.L1 and U2.L2: the density, computed by the augmented
ODE of U2.L2 (likelihood is not in the training loss; it is recovered here, at evaluation), and the image of a
grid of $x_0$ under the flow map. Put it beside your U2.L2 picture: at the budget's iterations they do not look
alike. Part 4 returns to this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
density_heatmap(make_log_prob(RESULTS["two-moons"]["field"]), ax=axes[0], n=120)
warped_grid(make_generator(RESULTS["two-moons"]["field"]), ax=axes[1])
for ax in axes:
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Part 3 — Sampling with the course's solver

A sample is the time-1 point of the solution of $\mathrm{d}x = u_t^\theta(x)\,\mathrm{d}t$ that starts at
$x_0 \sim \mathcal{N}(0, I_2)$ at $t = 0$. The solver is the given `common/odesolve.py`. Its result is a `SolverOutput`, a
`torch.Tensor` that carries `.nfe`, `.method`, `.n_steps`, `.accepted` and `.rejected`. The attributes do not
survive arithmetic: read them from the value the solver returned.

In [ ]:
@torch.no_grad()
def sample_cfm(field: nn.Module, n: int, *, seed: int, method: str = "adaptive_heun",
               n_steps: int | None = None, rtol: float | None = None, atol: float | None = None):
    """Draw n samples: x_0 ~ N(0, I_2), then integrate dx/dt = u_t^theta(x) from t = 0 to t = 1 (TODO 3.1).

    Uses the given `common/odesolve.py`. Returns the SolverOutput, which carries .nfe, .method,
    .n_steps, .accepted and .rejected: read them from this value, before any arithmetic.
    """
    # TODO 3.1: draw x0 from N(0, I_2), shape (n, 2), with torch.Generator().manual_seed(seed).
    # Then call odesolve.odeint with a right-hand side f(t, x), time first, from t = 0 to t = 1,
    # passing method, n_steps, rtol and atol through. Return what odeint returns, unchanged.
    raise NotImplementedError

In [ ]:
# Checkpoint 3.1: your sample_cfm on the reference weights
xs = sample_cfm(reference_field(), 4, seed=11, method="rk4", n_steps=8)
assert isinstance(xs, odesolve.SolverOutput) and getattr(xs, "nfe", None) is not None, \
    "return the SolverOutput of odeint itself: arithmetic on it (even 1.0 * x) drops .nfe"
assert xs.nfe == 32, f"NFE {xs.nfe}: rk4 with 8 steps calls the field 4 times per step"
assert torch.allclose(xs.as_subclass(torch.Tensor), torch.tensor([[0.4145822525024414, 2.1388368606567383], [-0.5966128706932068, -1.3736572265625], [-1.2931066751480103, 0.012481724843382835], [0.16029943525791168, 0.5841015577316284]]), atol=1e-5), \
    "the samples differ: x0 from torch.randn(n, 2) with the seeded generator, and integrate from t = 0 to t = 1"
for name, kw in (("rk4, 8 steps", {"method": "rk4", "n_steps": 8}),
                 ("adaptive Heun, rtol 1e-5", {"method": "adaptive_heun", "rtol": 1e-5, "atol": 1e-7})):
    x = sample_cfm(RESULTS["two-moons"]["field"], 4096, seed=11, **kw)
    print(f"{name:26s} {x.report()}")
print("Checkpoint 3.1 passed.")

**Prompt 3.1.** Compare the two reports above: the NFE of each method, and the accepted and rejected steps of
the adaptive one. On the reference computer the 8-step RK4 samples were within
1.4e-04 of a tight dopri5 solve. What does that tell you about the
tolerance a picture of samples needs?

### The evaluation sweep

The test NLL of the CFM model, by the augmented ODE of U2.L2, over the budget's tolerance grid. Same grid, same
test set, same solver (dopri5) and same exact divergence as the U2.L2 sweep, so panel (c) of the payoff plot
follows one protocol. The two fields are not of the same quality, and a smoother field can need fewer calls. One sentence from the course plan, now in code: likelihood abandoned during training,
recovered post hoc.

In [ ]:
def cfm_tolerance_rows(field: nn.Module, dataset: str, cfg: dict) -> list[dict]:
    """The evaluation sweep of the budget's tolerance grid (TODO 3.2): test NLL by the augmented ODE.

    The same grid, test set, solver (dopri5) and exact divergence as the U2.L2 sweep, so panel (c)
    follows one protocol (the two fields differ in quality). Likelihood is not in the CFM training loss; it is recovered here.
    """
    x_test = GENERATORS[dataset](cfg["data"]["n_test"], cfg["data"]["seed_test"])
    grid = cfg["tolerance_grid"]
    rows = []
    # TODO 3.2: for each rtol of grid["rtol"], with atol = rtol * grid["atol_factor"], call the given
    # evaluate_nll(field, x_test, rtol=..., atol=...) and append one row with the five keys of the
    # pain_tolerance schema: rtol, atol, nfe_eval, nll_test, wallclock_eval_s.
    raise NotImplementedError
    return rows

In [ ]:
# Checkpoint 3.2: your sweep on the reference weights reproduces the stored rows
rows = cfm_tolerance_rows(reference_field(), "two-moons", yaml.safe_load((ROOT / "configs" / "u2l2-budget.yaml").read_text()))
expected = [26, 32, 38,
            62, 86]
assert len(rows) == len(expected), f"{len(rows)} rows, expected one per rtol of the grid ({len(expected)})"
assert all(abs(r["nfe_eval"] - e) <= max(2, 0.1 * e) for r, e in zip(rows, expected)), \
    f"NFE over the grid is {[r['nfe_eval'] for r in rows]}, expected {expected}: one evaluate_nll call per rtol, atol = rtol * atol_factor"
assert abs(rows[-1]["nll_test"] - 3.403467893600464) < 1e-4, "the NLL at the tightest tolerance differs"
assert set(rows[0]) == set(painplots.SCHEMAS["pain_tolerance"]), "a row has exactly the five keys of the schema"
print("Checkpoint 3.2 passed.")

Now the sweep of YOUR two models, and the evidence directories of today, in the `painplots` v1.1 layout:
`cfm_evidence/<dataset>/` with the three CSV files, their PNGs and `meta.json`. The writer is given.

In [ ]:
def write_cfm_evidence(out_dir: Path, log: painplots.TrainLog, tol_rows: list[dict], meta: dict,
                       cfg: dict) -> list[str]:
    """Write one CFM evidence directory in the painplots v1.1 layout; return the validator's problems."""
    shutil.rmtree(out_dir, ignore_errors=True)
    painplots.pain_wallclock(log, out_dir)
    painplots.pain_nfe_growth(log, out_dir)           # two columns of zeros, by the NFE rule
    painplots.pain_tolerance(tol_rows, out_dir)
    painplots.write_meta(meta, out_dir)
    (out_dir / "resolved_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=True))
    return painplots.validate_evidence_dir(out_dir, expected_session="U3.L1")


for ds in ("two-moons", "checkerboard"):
    rs = RESULTS[ds]
    c = CFG[ds]
    rs["tol"] = cfm_tolerance_rows(rs["field"], ds, c)
    resolved = {**c, "dataset": ds}
    meta = painplots.make_meta(ds, c["seed"], resolved, pair_id=PAIR_ID, budget_version=c["budget_version"],
                               evidence_origin="student", divergence_mode="exact", session_id="U3.L1")
    problems = write_cfm_evidence(CFM_EVIDENCE / ds, rs["log"], rs["tol"], meta, resolved)
    print(f"[{ds}] validator:", "PASS" if not problems else problems)
    assert not problems

In [ ]:
# TODO 3.3: draw 4096 samples of RESULTS["checkerboard"]["field"] with your sample_cfm (seed 11, rk4,
# 32 steps) into xs, compute frac_out = the fraction OUTSIDE the support with in_checkerboard_support, and
# scatter the samples on EXTENT with equal axes. Print the solver report of the samples.
xs, frac_out = None, None
raise NotImplementedError

In [ ]:
# Checkpoint 3.3: a fraction, and the fraction OUTSIDE
assert frac_out is not None and 0.0 <= frac_out <= 1.0, "frac_out is a fraction of the samples, between 0 and 1"
inside = float(in_checkerboard_support(xs.as_subclass(torch.Tensor)).float().mean())
assert abs(frac_out - (1 - inside)) < 1e-9, "frac_out counts the samples OUTSIDE the squares, not inside"
print("Checkpoint 3.3 passed.")

**Prompt 3.3.** The flow map of a regular field is a homeomorphism, so a model built from a full-support base
has full support (U1.T2 §5), and its density is positive everywhere: it cannot have exact empty gaps between the
squares. Separate that statement
from what your picture shows at this resolution and after this budget. What does `frac_out` measure, and what
does it not?

## Part 4 — The payoff plot

The cell is given complete. For each dataset it chooses the pair (your own, or the reference), validates both
directories, and draws three panels on shared axes: (a) wall-clock per training iteration; (b) **training solver
NFE**, with the CFM line at 0 and its one direct field call per step shown beside it, not relabeled; (c) the
evaluation sweep, solver NFE against rtol. The CNF curves are drawn as measured.

In [ ]:
PAIRS = {}
for ds in ("two-moons", "checkerboard"):
    mode, pair = payoff.choose_pair(ds, own_cnf=EVIDENCE, own_cfm=CFM_EVIDENCE, ref_cnf=REF_CNF, ref_cfm=REF_CFM)
    PAIRS[ds] = (mode, pair)
    fig = payoff.plot_payoff(pair)
    display(fig)
    fig.savefig(CFM_EVIDENCE / f"payoff_{ds}.png", dpi=150, bbox_inches="tight")
    r = payoff.paired_ratios(pair)
    print(f"[{ds}] whole run: CNF {r['cnf_cum_s']:.0f} s, CFM {r['cfm_cum_s']:.1f} s; median ratio per iteration {r['median_s_per_iter_ratio']:.0f}x")
    print(payoff.tolerance_table(pair))

Same data family. Same field architecture. Same budget discipline. Paired machine. The U2 question, *can we
learn the transport field by regression without an ODE solve inside every training loss?*, is answered by the
receipts on your screen. The numerical solve has moved out of the CFM training objective. U3.T2 proves the exact
squared-loss identity that makes the conditional target legitimate.

### The two costs, read one last time

The table of U2.T2 (v3), with its CNF row opened by phase of computation and a CFM row beside it. Nothing is
struck out: the costs moved, they did not all disappear.

| Phase | CNF trained by likelihood (U2.L2) | CFM (today) |
|:--|:--|:--|
| training step | likelihood solve + adjoint solve, divergence at every call | one call of $u_t^\theta$, no solve: solver NFE 0 |
| sampling | an ODE solve, $t = 0 \to 1$ | an ODE solve, $t = 0 \to 1$ |
| density (optional) | an augmented ODE solve, divergence at every call | the same augmented solve, after learning |

### A second receipt: what the saved time buys

Panel (a) and the NLL column above say two things at once. Each CFM step is far cheaper. At the frozen budget's
3000 iterations (rescue budget: 1500), the CFM model is also **weaker** than the CNF: on the reference computer its
two-moons test NLL was 3.30 to 3.44 nats over five
seeds, against 2.10 to 2.19 for the CNF.
Equal iterations is not equal cost. The cell below trains CFM for ten times the budget's iterations (the
warmup-cosine schedule stretches with them), and changes nothing else. This run is **outside the budget**: it writes no evidence and pairs with
nothing. In own mode the cell still prints both training times: they come from this one computer (one
machine id, one thread), which is the only reason they stand side by side, unless Part 4 printed a device
warning. In reference mode it prints no time comparison.

In [ ]:
RUN_LONG = True             # about ten times the training time of Part 2, for each dataset
LONG = {}
if RUN_LONG:
    for ds in ("two-moons", "checkerboard"):
        c = CFG[ds]
        f_long, log_long = train_cfm(c, ds, iters=10 * c["train"]["iters"], verbose=False)
        x_test = GENERATORS[ds](c["data"]["n_test"], c["data"]["seed_test"])
        nll = evaluate_nll(f_long.eval(), x_test, rtol=1e-6, atol=1e-8)["nll"]
        mode, pair = PAIRS[ds]
        cnf_nll = pair.cnf["pain_tolerance"]["nll_test"][-1]
        LONG[ds] = {"nll": nll, "train_s": sum(log_long.step_s)}
        print(f"[{ds}] CFM, {len(log_long)} iterations ({c['budget_version']}): test NLL {nll:.3f}; "
              f"CNF of the {mode} pair ({pair.key[2]}, one seed): test NLL {cnf_nll:.3f}")
        if mode == "own":          # the same computer: the wall-clock times may be compared
            cnf_s = pair.cnf["pain_wallclock"]["cum_wallclock_s"][-1]
            devices = "" if pair.cnf["meta"]["device_name"] == pair.cfm["meta"]["device_name"] else \
                " (different devices: see the Part 4 warning)"
            print(f"[{ds}] training time on this computer: CFM {sum(log_long.step_s):.0f} s, CNF {cnf_s:.0f} s{devices}")
        else:                      # another computer: no wall-clock comparison (the pairing rule)
            print(f"[{ds}] training time: CFM {sum(log_long.step_s):.0f} s on this computer; the reference CNF ran on "
                  "another computer, so the two times are not compared")

On the reference computer, over seeds 2026 to 2030, the 30000-iteration
runs took 7.2% of the mean CNF training time on two-moons. Their test
NLL was 2.184 to 2.206 on two-moons (CNF:
2.100 to 2.195) and
3.333 to 3.355 on the checkerboard (CNF:
3.300 to 3.330).

**Prompt 4.** Write two sentences about your own numbers: one about cost per training step, one about model
quality at equal iterations and after ten times the iterations (on the reference computer, about 7% of the CNF's
training time). Name the pair (own or reference) each number comes from.

## Part 5 — TorchCFM cross-check, and the JAX moment

Libraries are cross-checks, never substitutes. TorchCFM (Tong et al.) implements the same CondOT construction in
`ConditionalFlowMatcher`. Give it **our** $t$, $x_0$, $x_1$, and it must return our $x_t$ and our target.

In [ ]:
matcher = ConditionalFlowMatcher(sigma=0.0)          # sigma = 0: the CondOT path, no added noise (import: prelude)
print(inspect.signature(matcher.sample_location_and_conditional_flow))

In [ ]:
def torchcfm_batch_fn(matcher):
    """A `batch_fn` for `train_cfm` that hands OUR (t, x0, x1) to TorchCFM and uses its x_t and target."""
    def batch_fn(train_x: torch.Tensor, batch: int, gen: torch.Generator):
        t, x0, x1, _, _ = sample_batch(train_x, batch, gen)
        # TODO 5.1: hand x0, x1 AND our t to matcher.sample_location_and_conditional_flow (its signature is
        # printed above). It returns (t, xt, ut): take t, xt and target = ut from it.
        raise NotImplementedError
        return t, x0, x1, xt, target
    return batch_fn

In [ ]:
# Checkpoint 5.1: the same inputs give the same outputs; the same training gives the same losses
bf = torchcfm_batch_fn(matcher)
a = sample_batch(train_x, 256, torch.Generator().manual_seed(3))
b = bf(train_x, 256, torch.Generator().manual_seed(3))
for name, i in (("t", 0), ("x_t", 3), ("target", 4)):
    assert torch.allclose(a[i], b[i], atol=1e-6), f"{name} differs between your code and TorchCFM's"
_, log_t = train_cfm(cfg, "two-moons", batch_fn=bf, verbose=False)
diff = max(abs(x - y) for x, y in zip(RESULTS["two-moons"]["log"].loss, log_t.loss))
print(f"largest difference between the two loss curves over {len(log_t)} iterations: {diff:.2e}")
assert diff < 1e-4, "the two runs should see the same numbers at every step"
print("Checkpoint 5.1 passed.")

### The source, read

The matcher class is short. The cell prints the six methods that today's loop reaches. Fill the two blank rows of
the table.

In [ ]:
for name in ("compute_mu_t", "compute_sigma_t", "sample_xt", "sample_noise_like", "compute_conditional_flow",
             "sample_location_and_conditional_flow"):
    print(inspect.getsource(getattr(ConditionalFlowMatcher, name)))

**TODO 5.2.** Fill the two blank rows.

| Your line | TorchCFM |
|:--|:--|
| `t = torch.rand(batch, generator=gen, ...)` | `t = torch.rand(x0.shape[0]).type_as(x0)` when `t` is `None` |
| `xt = t[:, None] * x1 + (1 - t[:, None]) * x0` | |
| `target = x1 - x0` | |

**TODO 5.3 (reserved).** This cell is left empty on purpose. U3.L3 reads TorchCFM's optimal-transport matcher
here, beside the class you just read.

### The JAX moment (take-home by default)

The same loop in plain JAX: the field, the CondOT minibatch, the loss, and AdamW with the warmup-cosine schedule
written by hand (no optax, as in the JAX primer). Honest timing: the up-front price is measured from before
`lower`, and lowering and compilation are reported separately from the steady state.

In [ ]:
RUN_JAX = False             # take-home: set True to run (3 s on the reference computer)

import jax
import jax.numpy as jnp

def jax_cfm(cfg: dict, train_x: np.ndarray, seed: int, iters: int | None = None) -> dict:
    """The same loop in plain JAX: the budget's MLP, tanh, a zero head, AdamW and warmup-cosine by hand.

    Honest timing (U0.L5): the up-front price is measured from BEFORE `lower`, and lowering and XLA
    compilation are recorded separately; the steady state is a median over calls, each closed by
    `block_until_ready`.
    """
    iters = cfg["train"]["iters"] if iters is None else iters
    m, o, b = cfg["model"], cfg["opt"], cfg["train"]["batch"]
    key = jax.random.key(seed)
    sizes = [3] + [m["hidden"]] * m["depth"] + [2]
    params = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        key, k1, k2 = jax.random.split(key, 3)
        bound = 1.0 / math.sqrt(n_in)            # the PyTorch nn.Linear default, U(-1/sqrt(n), 1/sqrt(n))
        w = jax.random.uniform(k1, (n_in, n_out), minval=-bound, maxval=bound)
        bias = jax.random.uniform(k2, (n_out,), minval=-bound, maxval=bound)
        if i == len(sizes) - 2:
            w, bias = jnp.zeros_like(w), jnp.zeros_like(bias)     # the zero head of the budget
        params.append((w, bias))

    def field(params, t, x):                       # u_t^theta(x), t of shape (B,)
        h = jnp.concatenate([x, t[:, None]], axis=1)
        for w, bias in params[:-1]:
            h = jnp.tanh(h @ w + bias)
        w, bias = params[-1]
        return h @ w + bias

    def loss_fn(params, t, x0, x1):
        xt = t[:, None] * x1 + (1 - t[:, None]) * x0
        return jnp.mean(jnp.sum((field(params, t, xt) - (x1 - x0)) ** 2, axis=1))

    warm = max(1, round(cfg["sched"]["warmup_frac"] * iters))
    b1, b2, eps, wd, lr0 = o["betas"][0], o["betas"][1], 1e-8, o["weight_decay"], o["lr"]

    def lr_at(step):                               # warmup-cosine on the step index, as in PyTorch
        p = (step - warm) / max(1, iters - warm)
        return lr0 * jnp.where(step < warm, step / warm, 0.5 * (1 + jnp.cos(jnp.pi * p)))

    def step_fn(params, mom, vel, step, key, data):
        k1, k2, k3, key = jax.random.split(key, 4)
        idx = jax.random.randint(k1, (b,), 0, data.shape[0])
        x1 = data[idx]
        x0 = jax.random.normal(k2, x1.shape)
        t = jax.random.uniform(k3, (b,))
        loss, grads = jax.value_and_grad(loss_fn)(params, t, x0, x1)
        lr = lr_at(step)
        n = step + 1
        mom = jax.tree.map(lambda m_, g: b1 * m_ + (1 - b1) * g, mom, grads)
        vel = jax.tree.map(lambda v_, g: b2 * v_ + (1 - b2) * g * g, vel, grads)
        params = jax.tree.map(                     # AdamW: decoupled weight decay, then the Adam step
            lambda p_, m_, v_: p_ * (1 - lr * wd) - lr * (m_ / (1 - b1 ** n)) / (jnp.sqrt(v_ / (1 - b2 ** n)) + eps),
            params, mom, vel)
        return params, mom, vel, key, loss

    data = jnp.asarray(train_x)
    mom = jax.tree.map(jnp.zeros_like, params)
    vel = jax.tree.map(jnp.zeros_like, params)
    t0 = time.perf_counter()
    lowered = jax.jit(step_fn).lower(params, mom, vel, 0, key, data)
    t1 = time.perf_counter()
    compiled = lowered.compile()
    t2 = time.perf_counter()
    losses, times = [], []
    for step in range(iters):
        s0 = time.perf_counter()
        params, mom, vel, key, loss = compiled(params, mom, vel, step, key, data)
        loss.block_until_ready()
        times.append(time.perf_counter() - s0)
        losses.append(float(loss))
    return {"lower_s": t1 - t0, "compile_s": t2 - t1, "upfront_s": t2 - t0,
            "steady_s_per_iter": statistics.median(times[10:]), "losses": losses}


if RUN_JAX:
    r = jax_cfm(cfg, train_x.numpy(), cfg["seed"])
    print(f"lower {r['lower_s']:.2f} s + compile {r['compile_s']:.2f} s = up-front {r['upfront_s']:.2f} s; "
          f"steady {1e3 * r['steady_s_per_iter']:.2f} ms per step")
    print(f"mean loss of the last 300: JAX {statistics.fmean(r['losses'][-300:]):.3f}, "
          f"PyTorch {statistics.fmean(RESULTS['two-moons']['log'].loss[-300:]):.3f}")

## Part 6 — Freeze, log, and PS2.6

Log both runs to MLflow (local, `sqlite:///mlflow.db` next to this notebook, experiment `U3`), with the payoff
figure. `field_evals_train = 1` is logged as its own metric.

In [ ]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
for ds in ("two-moons", "checkerboard"):
    meta = json.loads((CFM_EVIDENCE / ds / "meta.json").read_text())
    run_id = painplots.log_run(None, run_name=f"u3l1-{ds}", log=RESULTS[ds]["log"], experiment="U3",
                               tags={"session": "U3.L1", "dataset": ds, "machine_id": meta["machine_id"],
                                     "pair_id": meta["pair_id"], "evidence_origin": "student",
                                     "pair_mode": PAIRS[ds][0]},
                               params={"budget_version": CFG[ds]["budget_version"], "seed": CFG[ds]["seed"],
                                       "iters": CFG[ds]["train"]["iters"], "config_hash": meta["config_hash"]})
    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifacts(str(CFM_EVIDENCE / ds), artifact_path="cfm_evidence")   # not under "pain_evidence"
        mlflow.log_metric("field_evals_train", 1)
        mlflow.log_figure(payoff.plot_payoff(PAIRS[ds][1]), f"payoff_{ds}.png")
    painplots.export_run(run_id, ROOT / "cfm_mlflow_export" / ds)
    print(f"[{ds}] run {run_id} logged and exported to cfm_mlflow_export/{ds}/")

### Freeze `cfm-scratch`

Your loop, your field and your sampling path are the artifact `cfm-scratch`. U3.L2 edits this code (it swaps the
path and the target), and U3.L3 imports its sampling utilities. Keep it clean, commit it, and tag it:

```bash
git add labs/lab-u3l1.ipynb cfm_evidence cfm_mlflow_export
git commit -m "U3.L1: cfm-scratch"
git tag u3l1-freeze
```

### PS2.6

*Deliverable: this lab, packaged. Course norm: correctness + reproducibility, not sample quality.*

- (a) Completed notebook, all TODOs, checkpoint cells executed.
- (b) `cfm_evidence/two-moons/` and `cfm_evidence/checkerboard/` complete under `painplots` v1.1 metadata (validator PASS).
- (c) The payoff plot rendered from the student's own valid v1.1 pair (or the shipped reference pair), exported via `mlflow.log_figure`.
- (d) Written answers (2–4 sentences each, course notation): (i) why do the solver-NFE fields read zero for CFM training while `field_evals_train=1` is logged separately, and what structurally was removed? (ii) read your eval tolerance/NFE panel without assuming monotonic laws; (iii) the promissory note names a theorem — state, in one sentence of plain prose, what relation it must establish between the two squared objectives.
- (e) `mlruns/` export for both runs.

The deadline is in the course calendar. For (e), the export of the cell above (`cfm_mlflow_export/`) is the
`mlruns/` export: the tracking database itself holds absolute paths and is never what you submit.

**PS2.6 (d)(i).**

**PS2.6 (d)(ii).**

**PS2.6 (d)(iii).**